# Linear Regression with HDB Geocoding

This notebook cleans raw rental records, joins HDB block attributes by a normalized address key, then uses the repository's postal-code geocoder from `src/transformers.py`. The geocoder derives postal sector, district, and region from `POSTAL_CODE` using `src/config/config_2026.yaml`.

## Join and geocoding design

A deterministic `TOWN | BLOCK | STREET` key is used because both files describe the same HDB block identity. The HDB reference is deduplicated and joined with `validate='many_to_one'`; this prevents accidental row multiplication. Coordinates and block attributes come from the HDB reference. Postal geocoding is applied after the join because postal code exists in the HDB file, not in the rental file. Invalid postal codes remain missing and are tracked through coverage diagnostics.

In [17]:
from pathlib import Path
import sys
import yaml
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / 'data' / 'train.csv').exists()), HERE)
GEO_ROOT = PROJECT_ROOT / 'models' / 'basic models' / 'linear regression + geo encoding'
GEO_SRC = GEO_ROOT / 'src'
sys.path.insert(0, str(GEO_ROOT))
from src.transformers import FloorTransformer, PostcodeRegionTransformer
with open(GEO_SRC / 'config' / 'config_2026.yaml', encoding='utf-8') as handle:
    geo_config = yaml.safe_load(handle)
postcode_geocoder = PostcodeRegionTransformer(geo_config)
floor_transformer = FloorTransformer(geo_config)
rent_raw = pd.read_csv(PROJECT_ROOT / 'data' / 'train.csv')
hdb_raw = pd.read_csv(PROJECT_ROOT / 'data' / 'sg-hdb-block.csv', dtype={'POSTAL_CODE': 'string'})
print('Rental data:', rent_raw.shape)
print('HDB reference:', hdb_raw.shape)

Rental data: (150000, 11)
HDB reference: (9464, 11)


## Cleaning

Text values are stripped, whitespace-collapsed, and lowercased. Common street-type spellings are canonicalized and flat-type variants are harmonized. `FURNISHED` is retained as a binary variable. The same function is applied to both datasets before constructing the join key.

In [18]:
STREET_REPLACEMENTS = {r'\bavenue\b':'ave', r'\broad\b':'rd', r'\bstreet\b':'st', r'\bdrive\b':'dr', r'\bcentral\b':'ctrl', r'\bplace\b':'pl', r'\bclose\b':'cl', r'\bcrescent\b':'cres', r'\blane\b':'ln', r'\bparkway\b':'pkwy', r'\bterrace\b':'ter', r'\bview\b':'vw'}
def normalize_text(s):
    return s.astype('string').str.strip().str.lower().str.replace(r'\s+', ' ', regex=True)
def clean_common(frame):
    out = frame.copy()
    for col in out.select_dtypes(include=['object','string']).columns: out[col] = normalize_text(out[col])
    if 'STREET' in out:
        for pattern, replacement in STREET_REPLACEMENTS.items(): out['STREET'] = out['STREET'].str.replace(pattern, replacement, regex=True)
        out['STREET'] = out['STREET'].str.replace(r'\s+', ' ', regex=True).str.strip()
    if 'FLAT_TYPE' in out: out['FLAT_TYPE'] = out['FLAT_TYPE'].replace({'1 room':'1-room','2 room':'2-room','3 room':'3-room','4 room':'4-room','5 room':'5-room'})
    if 'FURNISHED' in out: out['FURNISHED'] = out['FURNISHED'].replace({'yes':1,'y':1,'true':1,'1':1,'no':0,'n':0,'false':0,'0':0}).astype('Int64')
    return out
rent = clean_common(rent_raw).drop_duplicates().reset_index(drop=True)
hdb = clean_common(hdb_raw)
rent['RENT_APPROVAL_DATE'] = pd.to_datetime(rent['RENT_APPROVAL_DATE'], format='%Y-%m', errors='coerce')
rent['RENT_YEAR'] = rent['RENT_APPROVAL_DATE'].dt.year
rent['RENT_MONTH'] = rent['RENT_APPROVAL_DATE'].dt.month
rent['RENT_QUARTER'] = rent['RENT_APPROVAL_DATE'].dt.quarter
rent['LEASE_AGE'] = rent['RENT_YEAR'] - pd.to_numeric(rent['LEASE_COMMENCE_DATE'], errors='coerce')
rent['ADDRESS_KEY'] = rent['TOWN'] + '|' + rent['BLOCK'] + '|' + rent['STREET']
hdb['ADDRESS_KEY'] = hdb['TOWN'] + '|' + hdb['BLOCK'] + '|' + hdb['STREET']

In [19]:
hdb = hdb.drop_duplicates('ADDRESS_KEY', keep='first')
geo_columns = ['ADDRESS_KEY','POSTAL_CODE','LATITUDE','LONGITUDE','MAX_FLOOR','YEAR_COMPLETED','SUBZONE','PLANNING_AREA','REGION']
rent_geo = rent.merge(hdb[geo_columns], on='ADDRESS_KEY', how='left', validate='many_to_one')
rent_geo['HDB_BLOCK_MATCHED'] = rent_geo['LATITUDE'].notna().astype('int8')
rent_geo['BUILDING_AGE'] = rent_geo['RENT_YEAR'] - pd.to_numeric(rent_geo['YEAR_COMPLETED'], errors='coerce')
rent_geo = postcode_geocoder.transform(rent_geo)
rent_geo = floor_transformer.transform(rent_geo)
print('HDB match rate:', f"{rent_geo['HDB_BLOCK_MATCHED'].mean():.2%}")
print('Postal geocoding coverage:', f"{rent_geo['POSTAL_DISTRICT'].notna().mean():.2%}")
rent_geo[['POSTAL_CODE','POSTAL_SECTOR','POSTAL_DISTRICT','POSTAL_REGION','LATITUDE','LONGITUDE']].head()

HDB match rate: 100.00%
Postal geocoding coverage: 96.50%


,POSTAL_CODE,POSTAL_SECTOR,POSTAL_DISTRICT,POSTAL_REGION,LATITUDE,LONGITUDE
0,821604,82,19,3,1.404310,103.907413
1,543290,54,19,3,1.397018,103.897535
2,270005,27,10,1,1.312693,103.788309
3,792436,79,28,3,1.392878,103.876976
4,752467,75,27,3,1.451141,103.813630


## Model and evaluation

The postal code itself is excluded to avoid treating it as a continuous numeric measurement. It is transformed into `POSTAL_SECTOR`, `POSTAL_DISTRICT`, and `POSTAL_REGION` by `PostcodeRegionTransformer`. These derived geography labels are nominal categories, so they are one-hot encoded: each category becomes a 0/1 indicator and the regression model does not assume that district 20 is twice district 10.

Latitude and longitude are retained as continuous numeric variables because distance and location gradients are meaningful. `MAX_FLOOR`, `FLOOR_NUM`, and `FLOOR_NUM_CAPPED` are also numeric because they represent ordered quantities. `FLOOR_BAND` is one-hot encoded because it represents named ranges rather than a precise linear effect. `IS_HIGH_FLOOR`, `IS_BASEMENT`, `FURNISHED`, and `HDB_BLOCK_MATCHED` are binary 0/1 indicators and remain numeric.

All numeric columns use median imputation, while categorical columns use the most frequent category followed by `OneHotEncoder(handle_unknown='ignore')`. These transformations are inside a scikit-learn `Pipeline` and `ColumnTransformer`, so they are fitted only on the training split and applied unchanged to the test split. This prevents information from the evaluation data influencing the encoding.

In [20]:
# Encoding choices: coordinates, floor numbers, building age, and binary flags
# remain numeric. Postal sector/district/region and FLOOR_BAND are nominal
# categories, so one-hot encoding avoids imposing a false linear ordering.
y = pd.to_numeric(rent_geo.pop('MONTHLY_RENT'), errors='coerce')
X = rent_geo.drop(columns=['RENT_APPROVAL_DATE','ADDRESS_KEY','FEE','POSTAL_CODE'], errors='ignore')

valid = y.notna(); X, y = X.loc[valid].copy(), y.loc[valid].copy()
X = X.replace(r'^\s*$', np.nan, regex=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
numeric_geo = ['LATITUDE', 'LONGITUDE', 'MAX_FLOOR', 'YEAR_COMPLETED', 'BUILDING_AGE', 'FLOOR_NUM', 'FLOOR_NUM_CAPPED']
categorical_geo = ['POSTAL_SECTOR', 'POSTAL_DISTRICT', 'POSTAL_REGION', 'FLOOR_BAND']
for col in categorical_geo:
    if col in X: X[col] = X[col].astype('string')
num = X_train.select_dtypes(include=['number']).columns.tolist(); cat = X_train.select_dtypes(exclude=['number']).columns.tolist()
prep = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)])
model = Pipeline([('preprocessor', prep), ('regressor', LinearRegression())])
model.fit(X_train, y_train); predictions = model.predict(X_test)
metrics = pd.DataFrame({'metric':['MAE','RMSE','R²'], 'value':[mean_absolute_error(y_test,predictions), np.sqrt(mean_squared_error(y_test,predictions)), r2_score(y_test,predictions)]})
print(f'Train rows: {len(X_train):,}; test rows: {len(X_test):,}')
metrics

Train rows: 119,487; test rows: 29,872


,metric,value
0,MAE,391.021909
1,RMSE,510.842730
2,R²,0.536473
